<a href="https://colab.research.google.com/github/vikramvundyala/python_AI-ML/blob/main/Demo_Bagging_Iris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Certification in AIML
## A Program by IIIT-H and TalentSprint
## Not for grading

## Dataset

#### Description
The Iris dataset consists of 150 data instances. There are 3 classes (Iris Versicolor, Iris Setosa and Iris Virginica) each have 50 instances.


For each flower we have the below data attributes

- sepal length in cm
- sepal width in cm
- petal length in cm
- petal width in cm

To make our experiment easy we rename the classes  with numbers :

    "0": setosa
    "1": versicolor
    "2": virginica

## Setup Steps

In [ ]:
#@title Please enter your registration id to start: { run: "auto", display-mode: "form" }
Id = "" #@param {type:"string"}


In [ ]:
#@title Please enter your password (normally your phone number) to continue: { run: "auto", display-mode: "form" }
password = "" #@param {type:"string"}


In [ ]:
#@title Run this cell to complete the setup for this Notebook
from IPython import get_ipython

ipython = get_ipython()

notebook= "Demo_Bagging_Iris" #name of the notebook
Answer = "Ungraded"
def setup():
#  ipython.magic("sx pip3 install torch")
    from IPython.display import HTML, display
    ipython.magic("sx wget https://cdn.talentsprint.com/aiml/Experiment_related_data/Iris.csv")
    display(HTML('<script src="https://dashboard.talentsprint.com/aiml/record_ip.html?traineeId={0}&recordId={1}"></script>'.format(getId(),submission_id)))
    print("Setup completed successfully")
    return

def submit_notebook():

    ipython.magic("notebook -e "+ notebook + ".ipynb")

    import requests, json, base64, datetime

    url = "https://dashboard.talentsprint.com/xp/app/save_notebook_attempts"
    if not submission_id:
      data = {"id" : getId(), "notebook" : notebook, "mobile" : getPassword()}
      r = requests.post(url, data = data)
      r = json.loads(r.text)

      if r["status"] == "Success":
          return r["record_id"]
      elif "err" in r:
        print(r["err"])
        return None
      else:
        print ("Something is wrong, the notebook will not be submitted for grading")
        return None

    elif getComplexity() and getAdditional() and getConcepts() and getComments():
      f = open(notebook + ".ipynb", "rb")
      file_hash = base64.b64encode(f.read())

      data = {"complexity" : Complexity, "additional" :Additional,
              "concepts" : Concepts, "record_id" : submission_id,
              "id" : Id, "file_hash" : file_hash,
              "feedback_experiments_input" : Comments, "notebook" : notebook}

      r = requests.post(url, data = data)
      r = json.loads(r.text)
      if "err" in r:
        print(r["err"])
        return None
      else:
        print("Your submission is successful.")
        print("Ref Id:", submission_id)
        print("Date of submission: ", r["date"])
        print("Time of submission: ", r["time"])
        print("View your submissions: https://learn-iiith.talentsprint.com/notebook_submissions")
        # print("For any queries/discrepancies, please connect with mentors through the chat icon in LMS dashboard.")
      return submission_id
    else: submission_id


def getAdditional():
  try:
    if not Additional:
      raise NameError
    else:
      return Additional
  except NameError:
    print ("Please answer Additional Question")
    return None
def getComments():
  try:
    if not Comments:
      raise NameError
    else:
      return Comments
  except NameError:
    print ("Please answer Comments Question")
    return None

def getComplexity():
  try:
    if not Complexity:
      raise NameError
    else:
      return Complexity
  except NameError:
    print ("Please answer Complexity Question")
    return None

def getConcepts():
  try:
    if not Concepts:
      raise NameError
    else:
      return Concepts
  except NameError:
    print ("Please answer Concepts Question")
    return None

def getId():
  try:
    return Id if Id else None
  except NameError:
    return None

def getPassword():
  try:
    return password if password else None
  except NameError:
    return None

submission_id = None
### Setup
if getPassword() and getId():
  submission_id = submit_notebook()
  if submission_id:
    setup()

else:
  print ("Please complete Id and Password cells before running setup")


## Import required packages

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import resample
from sklearn import metrics

### Load the data

In [ ]:
# Load the iris dataset
iris = pd.read_csv("Iris.csv")
iris = iris.drop("Id",axis=1)
iris.head()


In [ ]:
# Species from Iris dataset
iris.species.unique()

In [ ]:
# Convert the labels to numericals
converter = {"Iris-setosa":0, "Iris-versicolor": 1,"Iris-virginica":2}
iris["species"] = [converter [i] for i in iris["species"]]

In [ ]:
# Split the data into train and test data
train_data, test_data = train_test_split(iris, test_size=0.2, random_state=42)
len(train_data), len(test_data)

### Sampling with replacement

In [ ]:
# Function to create 5 subsets with replacement. nTimes = No. of Subsets; howmany = No. of samples in a subset

def select_samples(nTimes, howmany, train_data):
  subsets = []
  for i in range(nTimes):
    subset_i = resample(train_data, n_samples=howmany, replace=True)
    subsets.append(subset_i)

    # To find number of unique samples in a subset
    unique_samples = len(np.unique(subset_i, axis=0))

    # To find no. of repeated samples in a subset
    repeated_samples = len(subset_i)-len(np.unique(subset_i, axis=0))

    print("D%d has %d samples in which %d are unique samples and %d are repeated samples" %(i,len(subset_i), unique_samples, repeated_samples))
  return subsets

In [ ]:
# Call the above function to create 5 subsets for train data, each of size 120
subsets = select_samples(5, 120, train_data)

In [ ]:
# Initialize the Decision tree
decision_tree = DecisionTreeClassifier(max_depth=2)

In [ ]:
# Classify each subset using Decision tree
def DT_subset(train_data, test_data, model):

  # Extract features and labels of the train_data and test_data
  X_train = train_data.iloc[:,:-1]
  y_train = train_data.iloc[:, -1]
  X_test = test_data.iloc[:,:-1]
  y_test = test_data.iloc[:,-1]

  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  score = metrics.accuracy_score(y_pred, y_test)

  return model, score

In [ ]:
# Calculate accuracy for each subset
for i,each in enumerate(subsets):
    model, score = DT_subset(each, test_data, decision_tree)
    print("Accuracy for {} subset: {}".format(i, score))

In [ ]:
print("Actuals and Predictions of 30 test samples")
test_labels = test_data.iloc[:, -1].astype(int)

# Create a dictionary for storing the labels
labels_30 = {"actual_values": test_labels}

# Get the prediction labels of 30 samples for all subsets
for i in range(1,6):
  model,score = DT_subset(subsets[i-1], test_data, decision_tree)
  print("Subset_",i, "Accuracy is", score)
  test_features = test_data.iloc[:,:-1]
  y_pred30 = model.predict(test_features)
  pred_30 = y_pred30.astype(int)
  labels_30["subset"+ str(i)] = pred_30

# Create a dataframe of 30 test samples with actuals and predictions of all 5 subsets
df_test = pd.DataFrame(labels_30)
df_test

## Please answer the questions below to complete the experiment:

In [ ]:
#@title How was the experiment? { run: "auto", form-width: "500px", display-mode: "form" }
Complexity = "" #@param ["","Too Simple, I am wasting time", "Good, But Not Challenging for me", "Good and Challenging for me", "Was Tough, but I did it", "Too Difficult for me"]


In [ ]:
#@title If it was very easy, what more you would have liked to have been added? If it was very difficult, what would you have liked to have been removed? { run: "auto", display-mode: "form" }
Additional = "" #@param {type:"string"}

In [ ]:
#@title Can you identify the concepts from the lecture which this experiment covered? { run: "auto", vertical-output: true, display-mode: "form" }
Concepts = "" #@param ["","Yes", "No"]

In [ ]:
#@title  Text and image description/explanation and code comments within the experiment: { run: "auto", vertical-output: true, display-mode: "form" }
Comments = "" #@param ["","Very Useful", "Somewhat Useful", "Not Useful", "Didn't use"]


In [ ]:
#@title Run this cell to submit your notebook  { vertical-output: true }
try:
  if submission_id:
      return_id = submit_notebook()
      if return_id : submission_id =return_id
  else:
      print("Please complete the setup first.")
except NameError:
  print ("Please complete the setup first.")